# Validação 03 — Consulta e normalização do PubMed

## Goal

Comprovar que um plano de busca pode consultar as E-utilities oficiais do NCBI e produzir artigos com PMID, título, autores, periódico, data, DOI, URL e proveniência da consulta.

## Setup

Fonte: [documentação oficial das E-utilities](https://www.ncbi.nlm.nih.gov/books/NBK25499/). O notebook faz uma consulta real e limitada; os resultados podem mudar conforme a indexação do PubMed. `NCBI_EMAIL` e `NCBI_API_KEY` são opcionais e podem ser definidos no ambiente.

In [1]:
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from pprint import pprint
import os
import sys

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root / "src"))

from fatofake import (
    PubMedClient,
    prepare_search_plan,
    search_pubmed,
    validate_analysis_input,
)

executed_at = datetime.now(timezone.utc).isoformat()
print(f"Execução UTC: {executed_at}")

Execução UTC: 2026-09-23T13:17:57.618395+00:00


## Steps

Criamos uma entrada e um plano controlado com uma consulta em inglês. Depois usamos `ESearch` para localizar PMIDs e `ESummary` para recuperar e normalizar os metadados.

In [2]:
class ControlledQueryPlanner:
    def generate_queries(self, claim: str) -> list[str]:
        return ["coffee cancer risk systematic review"]

In [3]:
analysis_input = validate_analysis_input(
    "Tomar café aumenta o risco de câncer."
)
search_plan = prepare_search_plan(analysis_input, ControlledQueryPlanner())
client = PubMedClient(
    email=os.getenv("NCBI_EMAIL"),
    api_key=os.getenv("NCBI_API_KEY"),
)
pubmed_result = search_pubmed(search_plan, client, max_results_per_query=3)

In [4]:
print("Resumo da consulta:")
pprint([asdict(item) for item in pubmed_result.query_results])

print("\nArtigos normalizados:")
pprint([asdict(publication) for publication in pubmed_result.publications])

Resumo da consulta:
[{'query': 'coffee cancer risk systematic review', 'total_matches': 103}]

Artigos normalizados:
[{'authors': ('Poorolajal J',
              'Moradi L',
              'Mohammadi Y',
              'Cheraghi Z',
              'Gohari-Ensaf F'),
  'doi': '10.4178/epih.e2020004',
  'journal': 'Epidemiology and health',
  'matched_queries': ('coffee cancer risk systematic review',),
  'pmid': '32023777',
  'publication_date': '2020 Feb 2',
  'source': 'PubMed',
  'title': 'Risk factors for stomach cancer: a systematic review and '
           'meta-analysis.',
  'url': 'https://pubmed.ncbi.nlm.nih.gov/32023777/'},
 {'authors': ('Wang J',
              'Qiu K',
              'Zhou S',
              'Gan Y',
              'Jiang K',
              'Wang D',
              'Wang H'),
  'doi': '10.1080/07853890.2025.2455539',
  'journal': 'Annals of medicine',
  'matched_queries': ('coffee cancer risk systematic review',),
  'pmid': '39834076',
  'publication_date': '2025 Jan 2

## Checks

As verificações confirmam que houve retorno real, que o limite foi respeitado e que cada registro possui os campos mínimos para rastreamento.

In [5]:
assert len(pubmed_result.query_results) == 1
assert pubmed_result.query_results[0].total_matches >= len(pubmed_result.publications)
assert 1 <= len(pubmed_result.publications) <= 3

for publication in pubmed_result.publications:
    assert publication.pmid.isdigit()
    assert publication.title
    assert publication.source == "PubMed"
    assert publication.url == f"https://pubmed.ncbi.nlm.nih.gov/{publication.pmid}/"
    assert search_plan.queries[0] in publication.matched_queries

print(f"Validação aprovada com {len(pubmed_result.publications)} artigos reais.")

Validação aprovada com 3 artigos reais.


## Next Steps

A integração inicial com o PubMed estará validada quando todas as células forem executadas sem erros. A próxima etapa será resolver e conferir a identidade dos artigos usando DOI e metadados de outras fontes.